# Amparex (SDAX) — wie viel Customer Support kann KI übernehmen?

Diese Analyse klassifiziert jedes SDAX-Ticket danach, **wie** es gelöst wurde, und leitet
daraus ab, welcher Anteil des Support-Aufkommens von KI unterstützt oder ganz
übernommen werden könnte.

**Drei Auflösungs-Buckets** (plus `unclear`, wenn der Text nicht ausreicht):

| Bucket | Bedeutung |
|---|---|
| `question_answer` | Frage, die aus Doku/Produktwissen beantwortbar ist — nichts muss geändert werden |
| `support_todo` | Konkrete Routine-Tätigkeit eines Support-Mitarbeiters im System |
| `team_handoff` | Braucht ein anderes Team: Bug, Feature, DB-Fix, Hardware, externer Dienstleister |

Dazu kommen pro Ticket:

- **`sub_bucket`** — feinere Aufteilung innerhalb des Buckets
- **`intent_label`** — freier kanonischer Intent in 2–5 deutschen Wörtern. Die *kleineren*
  Buckets werden nicht vorab geraten, sondern durch Aggregation dieses Labels entdeckt.
- **`ai_mode`** — `fully_automatable` / `ai_drafts_human_sends` / `ai_assisted_triage` / `human_only`.
  Bewusst **getrennt** vom Bucket: ein `support_todo` kann voll automatisierbar sein,
  während eine `question_answer` zur Kassenabrechnung trotzdem einen Menschen braucht.
- **`confidence`** und **`reasoning`** zum Stichproben-Check.

**Ablauf:** Refresh (1,5 Jahre SDAX) → Aufbereitung → Pilot (60 Tickets) → Vollklassifikation
→ Validierung gegen harte Eskalations-Signale → Auswertung → Management-Summary.

**Voraussetzungen:** `.env` im Projektverzeichnis mit

```
JIRA_URL= / JIRA_USERNAME= / JIRA_PASSWORD=
AZURE_OPENAI_API_KEY=
AZURE_OPENAI_ENDPOINT=https://<resource>.openai.azure.com/
AZURE_OPENAI_DEPLOYMENT_NAME=<Name des Deployments, nicht des Modells>
AZURE_OPENAI_API_VERSION=2024-10-21   # optional
```

Das Deployment muss auf einem Modell mit Structured Outputs laufen
(`gpt-4o-2024-08-06` oder neuer, `gpt-4o-mini` eingeschlossen). Ältere Deployments
lehnen das JSON-Schema ab — dann kommt jedes Ticket als `bad_request` zurück.

In [ ]:
import os
import subprocess
import sys
from datetime import date

import pandas as pd
import plotly.express as px
import plotly.io as pio
from dotenv import load_dotenv
from IPython.display import Markdown, display

%load_ext autoreload
%autoreload 2

load_dotenv(override=True)

pio.templates.default = "plotly_white"
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 220)

PROJECT = "SDAX"

# Fester Startpunkt statt relativem Fenster, damit der Datensatz eine saubere
# Kante hat und ein spaeterer Lauf denselben Zeitraum liefert.
START_DATE = date(2025, 1, 1)

# Die Daten sind bereits geholt (data/jira_data.pkl, 52.695 Tickets ab 01.01.2025).
# Der Pull dauert Stunden, deshalb ist er standardmaessig aus. Auf True setzen, um
# bis heute nachzuladen - backfill_jira.py checkpointet alle 100 Tickets, ein
# Abbruch kostet also hoechstens die letzten 100.
REFRESH = False
RESUME = True

required = (
    "JIRA_URL",
    "JIRA_USERNAME",
    "JIRA_PASSWORD",
    "AZURE_OPENAI_API_KEY",
    "AZURE_OPENAI_ENDPOINT",
    "AZURE_OPENAI_DEPLOYMENT_NAME",
)
for name in required:
    print(f"{name:26s} {'gesetzt' if os.getenv(name) else 'FEHLT'}")
print(
    f"{'AZURE_OPENAI_API_VERSION':26s} {os.getenv('AZURE_OPENAI_API_VERSION', '2024-10-21')}"
)
print(f"\nZeitraum ab: {START_DATE}   REFRESH={REFRESH}")

JIRA_URL                   gesetzt
JIRA_USERNAME              gesetzt
JIRA_PASSWORD              gesetzt
AZURE_OPENAI_API_KEY       gesetzt
AZURE_OPENAI_ENDPOINT      gesetzt
AZURE_OPENAI_DEPLOYMENT_NAME gesetzt
AZURE_OPENAI_API_VERSION   2024-10-21

Zeitraum ab: 2025-01-01   REFRESH=False


## 1 · Refresh — 1,5 Jahre SDAX aus Jira ziehen

Der Pull läuft über `backfill_jira.py` statt über eigenen Fetch-Code im Notebook. Das Skript
existiert genau für diesen Fall: es checkpointet die Rohdaten nach jedem Projekt, sichert die
bestehende `data/jira_data.pkl` vor dem Schreiben, und merged per Upsert — bereits
gesammelte Tickets gehen also nicht verloren.

Die Ausgabe wird live durchgereicht. Bei einem Abbruch einfach die Zelle erneut ausführen
(`RESUME = True` nutzt den Checkpoint und holt nur das fehlende Intervall nach).

In [2]:
if REFRESH:
    command = [
        sys.executable,
        "backfill_jira.py",
        "--project",
        PROJECT,
        "--start",
        START_DATE.isoformat(),
    ]
    if RESUME:
        command.append("--resume")
    print(" ".join(command), "\n", flush=True)

    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="", flush=True)
    code_ = process.wait()
    print(f"\nbackfill_jira.py beendet mit Exit-Code {code_}")
else:
    print("REFRESH=False - bestehende data/jira_data.pkl wird verwendet.")

REFRESH=False - bestehende data/jira_data.pkl wird verwendet.


## 2 · Laden und aufbereiten

Aus dem gemeinsamen Cache (SDIPR + SDAX + SDEU) nur die Amparex-Tickets herausziehen und
prüfen, wie viel Text überhaupt zur Verfügung steht. Das ist keine Formalie: Kommentare sind
das aussagekräftigste Feld, fehlen aber auf einem großen Teil der Tickets.

In [3]:
from data_loading import load_data

df_all = load_data()
if df_all is None or df_all.empty:
    raise RuntimeError(
        "data/jira_data.pkl ist leer oder fehlt - Abschnitt 1 mit REFRESH=True ausfuehren."
    )

df = df_all[df_all["key"].astype(str).str.startswith(f"{PROJECT}-")].copy()
df["created"] = pd.to_datetime(df["created"], errors="coerce", utc=True)
df = df.sort_values("created").reset_index(drop=True)

print(f"{len(df_all):,} Tickets im Cache, davon {len(df):,} {PROJECT}")
print(f"Zeitraum: {df['created'].min():%Y-%m-%d} -> {df['created'].max():%Y-%m-%d}")

64,282 Tickets im Cache, davon 52,695 SDAX
Zeitraum: 2025-01-01 -> 2026-09-16


### Kategorien wiederherstellen

Beim Refresh liefert die Jira-Assets-API **403 Forbidden**. `enrich_issue_assets` faengt das
ab, `asset_labels` bleibt leer, und `_asset_label` liefert fuer jedes Ticket `"Unbekannt"` —
alle Kategorienamen sind dadurch weg.

Verloren sind sie aber nicht: `main_category_id` / `sub_category_id` stehen weiterhin im
Datensatz, und `data/object_id_to_name.json` bildet diese IDs auf Namen ab. Damit werden 99,5 %
der Hauptkategorien wiederhergestellt, ohne die Assets-API zu brauchen.

Sobald die Assets-Berechtigung wieder da ist, ist diese Zelle ein No-Op — sie ersetzt nur
Labels, die tatsaechlich `Unbekannt` sind.

In [4]:
import json

with open("data/object_id_to_name.json", encoding="utf-8") as handle:
    OBJECT_ID_TO_NAME = json.load(handle)

MISSING_LABELS = {"", "nan", "None", "NA", "Unbekannt"}


def recover_category(frame, id_column, label_column):
    # Fuellt Labels, die die Assets-API nicht aufloesen konnte, aus der lokalen
    # id->name Map. Bereits gueltige Labels bleiben unangetastet.
    current = frame[label_column].astype(str).str.strip()
    ids = frame[id_column].astype(str).str.strip()
    recovered = ids.map(OBJECT_ID_TO_NAME)
    needs_fix = current.isin(MISSING_LABELS) & recovered.notna()
    return current.mask(needs_fix, recovered), int(needs_fix.sum())


for id_column, label_column in (
    ("main_category_id", "Hauptkategorie"),
    ("sub_category_id", "Unterkategorie"),
):
    df[label_column], fixed = recover_category(df, id_column, label_column)
    still_missing = df[label_column].isin(MISSING_LABELS).sum()
    print(
        f"{label_column:16s} {fixed:>6,} wiederhergestellt, "
        f"{still_missing:>6,} weiterhin ohne Namen, "
        f"{df[label_column].nunique()} verschiedene"
    )

Hauptkategorie   51,014 wiederhergestellt,  1,681 weiterhin ohne Namen, 23 verschiedene
Unterkategorie   13,742 wiederhergestellt, 38,953 weiterhin ohne Namen, 17 verschiedene


In [5]:
def fill_rate(series):
    text = series.astype(str).str.strip()
    return (~text.isin(["", "nan", "None", "[]", "NA"])).mean()


quality = pd.DataFrame(
    {
        "Feld": [
            "summary",
            "description",
            "comments",
            "Hauptkategorie",
            "Unterkategorie",
        ],
        "befuellt": [
            fill_rate(df[column])
            for column in [
                "summary",
                "description",
                "comments",
                "Hauptkategorie",
                "Unterkategorie",
            ]
        ],
        "Median Zeichen": [
            df[column].astype(str).str.len().median()
            for column in [
                "summary",
                "description",
                "comments",
                "Hauptkategorie",
                "Unterkategorie",
            ]
        ],
    }
)
quality["befuellt"] = (quality["befuellt"] * 100).round(1).astype(str) + " %"
display(quality)

print(f"\nHauptkategorien: {df['Hauptkategorie'].nunique()}")
print(f"Unterkategorien: {df['Unterkategorie'].nunique()}")

,Feld,befuellt,Median Zeichen
0,summary,100.0 %,28.0
1,description,80.5 %,99.0
2,comments,59.2 %,46.0
3,Hauptkategorie,100.0 %,12.0
4,Unterkategorie,100.0 %,9.0



Hauptkategorien: 23
Unterkategorien: 17


In [6]:
from ticket_classifier import build_ticket_text

df["ticket_text"] = [build_ticket_text(row) for _, row in df.iterrows()]
lengths = df["ticket_text"].str.len()

print("Laenge des Klassifikations-Texts pro Ticket (Zeichen):")
print(lengths.describe().round(0).to_string())
print(f"\nGeschaetzte Input-Tokens gesamt: ~{lengths.sum() / 4 / 1000:,.0f}k")

print("\n--- Beispiel ---\n")
print(df.loc[lengths.sort_values().index[len(df) // 2], "ticket_text"][:1200])

Laenge des Klassifikations-Texts pro Ticket (Zeichen):
count    52695.0
mean       749.0
std        991.0
min         41.0
25%        143.0
50%        320.0
75%        986.0
max       6015.0

Geschaetzte Input-Tokens gesamt: ~9,871k

--- Beispiel ---

Summary: Heilfürsorge BWL anlegen

Category: Stammdaten / Unbekannt

Description: Heilfürsorge BWL IK 103600923

wie anlegen? hat wohl was mit Polizei zu tun

Comments:
im Team gefragt
--------------------------------
hinterlegt mit https://amparex.atlassian.net/wiki/spaces/AP/pages/26254716/Abrechnung+mit+KT+-+Polizei


## 3 · Pilot — 60 Tickets, quer über alle Hauptkategorien

Bevor die ~10.000 Tickets durchlaufen, ein stratifizierter Pilot: zwei Tickets je
Hauptkategorie. Damit lässt sich Taxonomie und Prompt prüfen, ohne den vollen Lauf zu
bezahlen. Die `reasoning`-Spalte ist genau dafür da — kurz durchlesen, bevor es weitergeht.

In [7]:
from ticket_classifier import CACHE_PATH, classify_tickets

PILOT_CACHE = "data/sdax_classifications_pilot.parquet"

PER_CATEGORY = 2

# Grouping the key Series rather than the frame keeps Hauptkategorie in the
# result; include_groups=False would drop the very column we stratified on.
pilot_keys = (
    df.groupby("Hauptkategorie", dropna=False)["key"]
    .apply(lambda keys: keys.sample(min(len(keys), PER_CATEGORY), random_state=0))
    .tolist()
)
pilot = df[df["key"].isin(pilot_keys)].reset_index(drop=True)
print(
    f"Pilot: {len(pilot)} Tickets aus {pilot['Hauptkategorie'].nunique()} Hauptkategorien"
)

pilot_results = classify_tickets(pilot, cache_path=PILOT_CACHE, max_workers=8)

Pilot: 46 Tickets aus 23 Hauptkategorien
Classifying 38 tickets with gpt-5.6-luna (~5k input tokens, 8 workers).


  0%|          | 0/38 [00:00<?, ?ticket/s]

Done: 38 classified, 0 failed.
Tokens/ticket (mean over 38): in 1115, out 208 (davon reasoning 125)


In [8]:
pilot_view = pilot.merge(pilot_results, on="key", how="left")
display(
    pilot_view[
        [
            "key",
            "Hauptkategorie",
            "summary",
            "resolution_bucket",
            "sub_bucket",
            "intent_label",
            "ai_mode",
            "confidence",
            "reasoning",
        ]
    ].head(30)
)

print("\nVerteilung im Pilot:")
print(pilot_view["resolution_bucket"].value_counts(dropna=False).to_string())
print()
print(pilot_view["ai_mode"].value_counts(dropna=False).to_string())

,key,Hauptkategorie,summary,resolution_bucket,sub_bucket,intent_label,ai_mode,confidence,reasoning
0,SDAX-305826,AX Performance,"Hilfeanforderung: 'Nagengast, Gertrud \ Kunde'",support_todo,manual_workaround,amparex startet nicht,human_only,0.94,"After the Windows upgrade, Amparex no longer started and support resolved the issue by reinstalling the application on the affected work..."
1,SDAX-305899,Terminplaner,Termin löschen,question_answer,how_to,termin loeschberechtigung,fully_automatable,0.91,"The ticket concerns inconsistent appointment deletion rights and was addressed by explaining the relevant permission groups, without a c..."
2,SDAX-309185,Kasse,Schulung für Thema Kasse,question_answer,how_to,kassenbuchungen richtig durchfuehren,ai_drafts_human_sends,0.94,"The customer requests guidance on correctly performing cash-register bookings, which can be answered through product training or Academy..."
3,SDAX-312950,Buchhaltung,Hilfeanforderung: 'Stammdaten \ Kundenbindung Empfehlungsgutschein KwK \ Kundenbindung',support_todo,configuration_change,buchungskonto hinterlegen,ai_drafts_human_sends,0.96,"The request is to configure the accounting account for referral vouchers in the application, which is routine support work but affects f..."
4,SDAX-314898,Inventur,Inventur,question_answer,how_to,inventur fragen,ai_assisted_triage,0.82,The ticket indicates general questions about the inventory process but gives no specific question or requested system change.
5,SDAX-316294,AX Performance,AX startet nicht,team_handoff,hardware_or_infrastructure,ax keine serververbindung,ai_assisted_triage,0.91,"AX cannot connect to the server, and the suspected resolution requires an antivirus or infrastructure configuration change outside the n..."
6,SDAX-316468,Stammdaten,Automatischer Rabatt kann nicht entfernt werden,support_todo,configuration_change,automatischen rabatt entfernen,ai_drafts_human_sends,0.94,"The automatic discount setting needs to be changed so the discount is no longer applied, which is a routine configuration task with fina..."
7,SDAX-318137,Noah,Noah Link Wireless hat nicht funktiniert,team_handoff,hardware_or_infrastructure,noah link funktioniert nicht,ai_assisted_triage,0.78,The Noah Link Wireless problem concerns a hardware/driver setup issue and cannot be resolved solely through an Amparex application answer.
8,SDAX-324471,Lagerverwaltung,Charger wurde versehentlich zur Rep eingeschickt,question_answer,policy_or_process_clarification,versehentliche reparatursendung,ai_drafts_human_sends,0.90,"The ticket asks which process to follow for a charger sent to repair by mistake, without requesting a system change or technical fix."
9,SDAX-388324,Unbekannt,Tracer Verbindungsprogramm einrichten,team_handoff,hardware_or_infrastructure,tracer programm einrichten,human_only,0.94,"The request required installing and configuring the external Tracer connection program on the laptop, which is infrastructure-related wo..."



Verteilung im Pilot:
resolution_bucket
question_answer    15
team_handoff       11
support_todo       10
unclear            10

ai_mode
ai_assisted_triage       19
fully_automatable        10
ai_drafts_human_sends    10
human_only                7


## 4 · Vollklassifikation

Läuft über alle SDAX-Tickets. Ergebnisse landen in `data/sdax_classifications.parquet` und
werden alle 100 Tickets weggeschrieben — ein Abbruch kostet also höchstens die letzten 100.
Ein erneuter Lauf klassifiziert nur, was neu ist oder dessen Text sich geändert hat.

**Größenordnung — gemessen, nicht geschätzt.** Ein Probelauf über 12 typische Tickets auf
dem Deployment `gpt-5.6-luna` ergab im Mittel **1.023 Input-** und **182 Output-Token** pro
Ticket (davon 103 Reasoning-Token). Hochgerechnet auf ~35.000 Tickets:

| | |
|---|---|
| Input | ~35,8 Mio. Token |
| Output | ~6,4 Mio. Token |
| Laufzeit bei 16 Workern | ~2,5 Stunden |

Die Kosten hängen am Azure-Tarif für dieses Deployment — die Token-Mengen oben mit der
eigenen Preisliste multiplizieren. Der grösste Hebel wäre der System-Prompt: er macht rund
800 der 1.023 Input-Token aus. Azure cached Prefixes erst ab 1.024 Token automatisch, der
Prompt liegt also knapp darunter.

Die Zelle unten gibt vor dem Start die geschätzte Token-Zahl aus, und `classify_tickets`
meldet nach jedem Lauf den tatsächlichen Verbrauch — nach dem Pilot also gegenrechnen,
bevor der Volllauf startet.

In [9]:
results = classify_tickets(df, cache_path=CACHE_PATH, max_workers=16)
print(f"\n{len(results):,} Klassifikationen im Cache ({CACHE_PATH})")

Classifying 1 tickets with gpt-5.6-luna (~0k input tokens, 16 workers).


  0%|          | 0/1 [00:00<?, ?ticket/s]

Done: 0 classified, 1 failed.
Failure reasons:
bad_request    1

52,695 Klassifikationen im Cache (data/sdax_classifications.parquet)


/Users/ovm/.superset/worktrees/0c6e7603-a4ce-4b2b-b63e-a306002df9b9/loud-beast/ticket_classifier.py:429: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  merged = pd.concat(frames, ignore_index=True) if frames else fresh


In [10]:
labelled = df.merge(results.drop(columns=["model", "text_hash"]), on="key", how="left")

failed = labelled["resolution_bucket"].isna().sum()
if failed:
    print(
        f"WARNUNG: {failed} Tickets ohne Label - Zelle oben erneut ausfuehren (Retry)."
    )

BUCKET_ORDER = ["question_answer", "support_todo", "team_handoff", "unclear"]
AI_MODE_ORDER = [
    "fully_automatable",
    "ai_drafts_human_sends",
    "ai_assisted_triage",
    "human_only",
]
labelled["resolution_bucket"] = pd.Categorical(
    labelled["resolution_bucket"], categories=BUCKET_ORDER, ordered=True
)
labelled["ai_mode"] = pd.Categorical(
    labelled["ai_mode"], categories=AI_MODE_ORDER, ordered=True
)

print(f"{len(labelled):,} Tickets klassifiziert")

WARNUNG: 1 Tickets ohne Label - Zelle oben erneut ausfuehren (Retry).
52,695 Tickets klassifiziert


## 5 · Validierung gegen harte Signale

Die Klassifikation ist ein Modell-Urteil und muss gegengeprüft werden. Im Datensatz stecken
drei unabhängige, *strukturelle* Eskalations-Signale, die niemand für diese Analyse erzeugt
hat:

- **`has_exax_clone`** — Ticket wurde ins EXAX-Projekt geklont, also an die Entwicklung übergeben
- **`status`** `Frage an Fachabteilung` / `Antwort von Fachabteilung` — an eine Fachabteilung eskaliert
- **`resolution_band`** — wie lange es tatsächlich gedauert hat

Wenn `team_handoff` etwas taugt, muss die Klon-Rate in diesem Bucket deutlich höher liegen
als in den anderen. Tut sie das nicht, stimmt die Klassifikation nicht.

In [11]:
# NICHT has_exax_clone verwenden: data_transformation.load_issues_Amparex sucht
# nach Klonen im Projekt EXAX, tatsaechlich landen Klone aber in AXT. Die Spalte
# ist dadurch auf dem gesamten Datensatz konstant False und als Signal wertlos.
# Hier wird deshalb direkt auf clone_in_project geprueft, beide Schreibweisen.
ESCALATION_PROJECTS = ("AXT", "EXAX")
clone_targets = labelled["clone_in_project"].fillna("").astype(str)
labelled["cloned_to_dev"] = clone_targets.str.contains(
    "|".join(ESCALATION_PROJECTS), regex=True
)
labelled["escalated_structurally"] = labelled["cloned_to_dev"] | labelled[
    "status"
].isin(["Frage an Fachabteilung", "Antwort von Fachabteilung"])

print("Struktuelle Eskalations-Signale im Datensatz:")
print(
    f"  Klon nach {'/'.join(ESCALATION_PROJECTS)}: {labelled['cloned_to_dev'].sum():,}"
)
print(
    f"  Status Fachabteilung:      {labelled['status'].isin(['Frage an Fachabteilung', 'Antwort von Fachabteilung']).sum():,}"
)
print(
    f"  gesamt:                    {labelled['escalated_structurally'].sum():,} "
    f"({labelled['escalated_structurally'].mean():.1%} der Tickets)"
)
print(
    "\nDas Signal ist duenn (~1-2 %), taugt also als Richtungspruefung, nicht als\n"
    "Ground Truth: team_handoff muss klar oben liegen, exakte Werte sind Rauschen."
)

validation = (
    labelled.groupby("resolution_bucket", observed=False)
    .agg(
        Tickets=("key", "size"),
        Dev_Klon=("cloned_to_dev", "mean"),
        Fachabteilung=(
            "status",
            lambda s: s.isin(
                ["Frage an Fachabteilung", "Antwort von Fachabteilung"]
            ).mean(),
        ),
        strukturell_eskaliert=("escalated_structurally", "mean"),
        Median_Konfidenz=("confidence", "median"),
    )
    .reset_index()
)
for column in ["Dev_Klon", "Fachabteilung", "strukturell_eskaliert"]:
    validation[column] = (validation[column] * 100).round(1)

display(validation)

print(
    "\nLesart: die Zeile team_handoff muss bei 'strukturell_eskaliert' deutlich ueber den\n"
    "anderen Zeilen liegen. Ist der Unterschied klein, traegt die Klassifikation nicht."
)

Struktuelle Eskalations-Signale im Datensatz:
  Klon nach AXT/EXAX: 470
  Status Fachabteilung:      61
  gesamt:                    488 (0.9% der Tickets)

Das Signal ist duenn (~1-2 %), taugt also als Richtungspruefung, nicht als
Ground Truth: team_handoff muss klar oben liegen, exakte Werte sind Rauschen.


,resolution_bucket,Tickets,Dev_Klon,Fachabteilung,strukturell_eskaliert,Median_Konfidenz
0,question_answer,12479,0.5,0.0,0.5,0.93
1,support_todo,16612,0.5,0.0,0.5,0.91
2,team_handoff,12108,2.6,0.4,2.7,0.93
3,unclear,11495,0.1,0.0,0.1,0.98



Lesart: die Zeile team_handoff muss bei 'strukturell_eskaliert' deutlich ueber den
anderen Zeilen liegen. Ist der Unterschied klein, traegt die Klassifikation nicht.


In [12]:
band_column = (
    "resolution_band" if "resolution_band" in labelled.columns else "resolution"
)
duration = (
    pd.crosstab(labelled["resolution_bucket"], labelled[band_column], normalize="index")
    * 100
).round(1)
display(duration)

figure = px.imshow(
    duration,
    text_auto=True,
    aspect="auto",
    color_continuous_scale="Blues",
    labels={"x": "Bearbeitungsdauer", "y": "Bucket", "color": "% der Zeile"},
    title="Bearbeitungsdauer je Auflösungs-Bucket (%, zeilenweise)",
)
figure.show()

resolution_band,> 1 Arbeitstag,Nicht abgeschlossen,≤ 1 Arbeitstag,≤ 1 Stunde
resolution_bucket,,,,
question_answer,11.2,14.2,5.3,69.3
support_todo,12.5,10.9,5.7,70.8
team_handoff,17.0,18.0,5.7,59.3
unclear,8.0,3.6,4.5,84.0


In [13]:
# Stichprobe zum Mitlesen: drei Tickets je Bucket, mit Begruendung.
labelled_ok = labelled.dropna(subset=["resolution_bucket"])
sample_keys = (
    labelled_ok.groupby("resolution_bucket", observed=True)["key"]
    .apply(lambda keys: keys.sample(min(len(keys), 3), random_state=1))
    .tolist()
)
sample = labelled_ok[labelled_ok["key"].isin(sample_keys)]
display(
    sample[
        [
            "key",
            "resolution_bucket",
            "ai_mode",
            "Hauptkategorie",
            "summary",
            "reasoning",
        ]
    ].reset_index(drop=True)
)

,key,resolution_bucket,ai_mode,Hauptkategorie,summary,reasoning
0,SDAX-306773,support_todo,ai_drafts_human_sends,Terminplaner,Hausbesuchsicon und Termintyp anlegen,"The customer requests creation of a home-visit icon and appointment type in the appointment planner, which is a routine configuration ch..."
1,SDAX-306853,team_handoff,ai_assisted_triage,Abrechnung Kostenträger,"Hilfeanforderung: 'Spang, Joachim \ Angebote \ AN-01-0039-25'",The recurring inability to delete duplicated offer positions across multiple workstations indicates a reproducible software defect requi...
2,SDAX-322496,unclear,ai_assisted_triage,Hardware,TSE Meldepflicht,The ticket only records a call about the TSE reporting obligation and contains no specific question or requested action.
3,SDAX-391092,support_todo,ai_drafts_human_sends,Schnittstellen,API-S einrichten,"The customer requested activation of the API for two branches, a routine configuration change that support can perform or verify without..."
4,SDAX-393827,question_answer,fully_automatable,Sonstige Themen,gezeigt wo Barmer Vertrag hinterlegen,"The ticket asks where/how to enter a Barmer insurer contract in Amparex, which is a product-usage question requiring no system change."
5,SDAX-397529,question_answer,human_only,Lagerverwaltung,Frage zu Eingangsrechnung wie am besten hinterlegen,"The customer wants an explanation of how to record supplier incoming invoices in inventory management, which is answerable through produ..."
6,SDAX-405491,unclear,ai_assisted_triage,Stammdaten,Glaspreislisten nicht up to date,"The ticket only states that glass price lists are outdated and mentions a phone call, without specifying the required correction or next..."
7,SDAX-408819,team_handoff,ai_assisted_triage,Installation / Einrichtung,Installation AX PC 1,"The ticket requests installation of Amparex on a specific PC, which requires technical or infrastructure work beyond routine support."
8,SDAX-410180,unclear,ai_assisted_triage,Schnittstellen,TSE Cloud Herr Groß,"The ticket only references TSE Cloud and a search/import identifier, without describing a request or problem to resolve."
9,SDAX-412809,team_handoff,ai_assisted_triage,AX Performance,AX startet nicht - deinstalliert und neu installiert,"AX still fails to start after reinstallation, indicating a technical fault requiring investigation beyond routine support steps."


## 6 · Auswertung

In [14]:
bucket_counts = (
    labelled["resolution_bucket"]
    .value_counts(dropna=False)
    .rename_axis("Bucket")
    .reset_index(name="Tickets")
)
bucket_counts["Anteil %"] = (bucket_counts["Tickets"] / len(labelled) * 100).round(1)
display(bucket_counts)

figure = px.bar(
    bucket_counts,
    x="Bucket",
    y="Tickets",
    text="Anteil %",
    title=f"Auflösungs-Buckets über {len(labelled):,} SDAX-Tickets",
)
figure.update_traces(texttemplate="%{text} %", textposition="outside")
figure.show()

,Bucket,Tickets,Anteil %
0,support_todo,16612,31.5
1,question_answer,12479,23.7
2,team_handoff,12108,23.0
3,unclear,11495,21.8
4,NaN,1,0.0


In [15]:
ai_counts = (
    labelled["ai_mode"]
    .value_counts(dropna=False)
    .rename_axis("ai_mode")
    .reset_index(name="Tickets")
)
ai_counts["Anteil %"] = (ai_counts["Tickets"] / len(labelled) * 100).round(1)
display(ai_counts)

figure = px.bar(
    ai_counts,
    x="ai_mode",
    y="Tickets",
    text="Anteil %",
    color="ai_mode",
    title="Wie viel könnte KI übernehmen?",
)
figure.update_traces(texttemplate="%{text} %", textposition="outside")
figure.update_layout(showlegend=False)
figure.show()

,ai_mode,Tickets,Anteil %
0,ai_assisted_triage,22575,42.8
1,ai_drafts_human_sends,17378,33.0
2,fully_automatable,9892,18.8
3,human_only,2849,5.4
4,NaN,1,0.0


In [16]:
# Bucket x ai_mode: zeigt, dass die beiden Achsen wirklich Unterschiedliches messen.
cross = pd.crosstab(labelled["resolution_bucket"], labelled["ai_mode"])
display(cross)
figure = px.imshow(
    cross,
    text_auto=True,
    aspect="auto",
    color_continuous_scale="Blues",
    labels={"x": "ai_mode", "y": "Bucket", "color": "Tickets"},
    title="Auflösungs-Bucket × KI-Eignung",
)
figure.show()

ai_mode,fully_automatable,ai_drafts_human_sends,ai_assisted_triage,human_only
resolution_bucket,,,,
question_answer,9693,2446,266,74
support_todo,162,14854,944,652
team_handoff,0,77,10523,1508
unclear,37,1,10842,615


### 6.1 · Kategorien — das eigentliche Ergebnis

Welche Hauptkategorien bestehen überwiegend aus beantwortbaren Fragen und Routine-Aufgaben,
und welche aus echten Eskalationen? Das ist die Antwort auf „wo setzen wir KI zuerst an“.

In [17]:
TOP_N_CATEGORIES = 20

top_categories = labelled["Hauptkategorie"].value_counts().head(TOP_N_CATEGORIES).index
subset = labelled[labelled["Hauptkategorie"].isin(top_categories)]

share = (
    pd.crosstab(
        subset["Hauptkategorie"], subset["resolution_bucket"], normalize="index"
    )
    * 100
).round(1)
share = share.loc[top_categories]

figure = px.imshow(
    share,
    text_auto=True,
    aspect="auto",
    color_continuous_scale="Blues",
    labels={"x": "Bucket", "y": "Hauptkategorie", "color": "% der Kategorie"},
    title=f"Auflösungs-Bucket je Hauptkategorie (Top {TOP_N_CATEGORIES}, zeilenweise %)",
)
figure.update_layout(height=620)
figure.show()

In [18]:
category_summary = (
    labelled.groupby("Hauptkategorie", observed=True)
    .agg(
        Tickets=("key", "size"),
        frage_antwort=("resolution_bucket", lambda s: (s == "question_answer").mean()),
        support_todo=("resolution_bucket", lambda s: (s == "support_todo").mean()),
        team_handoff=("resolution_bucket", lambda s: (s == "team_handoff").mean()),
        voll_automatisierbar=("ai_mode", lambda s: (s == "fully_automatable").mean()),
        ki_entwurf=("ai_mode", lambda s: (s == "ai_drafts_human_sends").mean()),
    )
    .reset_index()
)
category_summary["KI_adressierbar"] = (
    category_summary["voll_automatisierbar"] + category_summary["ki_entwurf"]
)
category_summary["KI_adressierbare_Tickets"] = (
    category_summary["KI_adressierbar"] * category_summary["Tickets"]
).round(0)

for column in [
    "frage_antwort",
    "support_todo",
    "team_handoff",
    "voll_automatisierbar",
    "ki_entwurf",
    "KI_adressierbar",
]:
    category_summary[column] = (category_summary[column] * 100).round(1)

category_summary = category_summary.sort_values(
    "KI_adressierbare_Tickets", ascending=False
)
display(category_summary.head(25))

,Hauptkategorie,Tickets,frage_antwort,support_todo,team_handoff,voll_automatisierbar,ki_entwurf,KI_adressierbar,KI_adressierbare_Tickets
2,Abrechnung Kostenträger,9010,24.2,37.6,14.6,14.3,46.8,61.0,5500.0
11,Kunde,6671,31.4,37.9,15.3,23.4,45.2,68.6,4575.0
20,Stammdaten,6327,27.4,41.6,14.4,25.1,41.9,67.0,4238.0
16,Schnittstellen,3930,15.4,32.7,30.4,12.8,31.4,44.2,1737.0
19,Sonstige Themen,4070,20.1,16.8,17.3,16.4,16.5,32.9,1340.0
21,Terminplaner,1967,28.5,38.2,16.6,28.7,34.7,63.3,1246.0
6,Filialverwaltung / Berechtigungen,1631,23.8,49.3,7.9,21.0,50.3,71.3,1163.0
10,Kasse,1841,33.6,28.1,19.7,21.7,38.4,60.1,1106.0
5,Buchhaltung,1550,34.1,38.5,8.2,20.9,49.7,70.6,1095.0
12,Lagerverwaltung,1774,32.3,30.3,15.5,30.3,30.8,61.1,1084.0


### 6.2 · Kleinere Buckets — was sich aus den Intents ergibt

`intent_label` war im Prompt frei. Was hier oben steht, sind die real wiederkehrenden
Anfragetypen — die Kandidaten für einzelne KI-Automatisierungen, jeweils mit dem Anteil,
den das Modell für voll automatisierbar hält.

In [19]:
MIN_INTENT_COUNT = 5

intents = (
    labelled.dropna(subset=["intent_label"])
    .groupby("intent_label", observed=True)
    .agg(
        Tickets=("key", "size"),
        Bucket=(
            "resolution_bucket",
            lambda s: s.mode().iloc[0] if len(s.mode()) else None,
        ),
        voll_automatisierbar=("ai_mode", lambda s: (s == "fully_automatable").mean()),
        Top_Kategorie=(
            "Hauptkategorie",
            lambda s: s.mode().iloc[0] if len(s.mode()) else None,
        ),
    )
    .reset_index()
)
intents["voll_automatisierbar"] = (intents["voll_automatisierbar"] * 100).round(1)
intents = intents[intents["Tickets"] >= MIN_INTENT_COUNT].sort_values(
    "Tickets", ascending=False
)

print(
    f"{len(intents)} wiederkehrende Intents mit >= {MIN_INTENT_COUNT} Tickets, "
    f"zusammen {intents['Tickets'].sum():,} Tickets "
    f"({intents['Tickets'].sum() / len(labelled) * 100:.0f} % des Aufkommens)"
)
display(intents.head(40))

1105 wiederkehrende Intents mit >= 5 Tickets, zusammen 16,599 Tickets (32 % des Aufkommens)


,intent_label,Tickets,Bucket,voll_automatisierbar,Top_Kategorie
6233,cloud tse aktivieren,355,support_todo,0.3,Schnittstellen
4048,ax startet nicht,323,team_handoff,2.5,AX Performance
22670,rechnung stornieren,286,support_todo,4.5,Kunde
7246,dokumentvorlage anpassen,284,support_todo,4.9,Stammdaten
6253,cloud tse einrichten,203,support_todo,3.0,Schnittstellen
7658,drucker druckt nicht,185,team_handoff,0.5,Hardware
20271,noah startet nicht,174,team_handoff,1.7,Noah
1678,amparex startet nicht,160,team_handoff,1.2,AX Performance
23978,rueckruf anfordern,158,unclear,0.0,Sonstige Themen
15976,kostentraegerabrechnung korrigieren,126,support_todo,0.0,Abrechnung Kostenträger


In [20]:
top_intents = intents.head(25).sort_values("Tickets")
figure = px.bar(
    top_intents,
    x="Tickets",
    y="intent_label",
    color="Bucket",
    orientation="h",
    title="Top 25 wiederkehrende Anfragetypen",
)
figure.update_layout(height=700, yaxis_title=None)
figure.show()

### 6.3 · Aufwand statt Stückzahl

Tickets sind nicht gleich teuer. Gewichtet man das Volumen mit der tatsächlichen
Bearbeitungsdauer (Business-Stunden), verschiebt sich das Bild — ein seltener, aber teurer
Anfragetyp kann wichtiger sein als ein häufiger Einzeiler.

In [21]:
labelled["biz_hours"] = pd.to_numeric(
    labelled["time_to_resolution_biz_hours"], errors="coerce"
)

effort = (
    labelled.dropna(subset=["ai_mode"])
    .groupby("ai_mode", observed=True)
    .agg(
        Tickets=("key", "size"),
        Median_Stunden=("biz_hours", "median"),
        Summe_Stunden=("biz_hours", "sum"),
    )
    .reset_index()
)
effort["Anteil Tickets %"] = (effort["Tickets"] / effort["Tickets"].sum() * 100).round(
    1
)
effort["Anteil Aufwand %"] = (
    effort["Summe_Stunden"] / effort["Summe_Stunden"].sum() * 100
).round(1)
effort["Median_Stunden"] = effort["Median_Stunden"].round(2)
effort["Summe_Stunden"] = effort["Summe_Stunden"].round(0)
display(effort)

print(
    "\nHinweis: time_to_resolution_biz_hours misst die Zeit bis zum Statuswechsel, nicht die\n"
    "reine Bearbeitungszeit am Ticket. Als Groessenordnung brauchbar, nicht als Kostenrechnung."
)

,ai_mode,Tickets,Median_Stunden,Summe_Stunden,Anteil Tickets %,Anteil Aufwand %
0,fully_automatable,9892,0.0,86247.0,18.8,10.7
1,ai_drafts_human_sends,17378,0.0,207837.0,33.0,25.9
2,ai_assisted_triage,22575,0.0,459607.0,42.8,57.3
3,human_only,2849,0.0,48623.0,5.4,6.1



Hinweis: time_to_resolution_biz_hours misst die Zeit bis zum Statuswechsel, nicht die
reine Bearbeitungszeit am Ticket. Als Groessenordnung brauchbar, nicht als Kostenrechnung.


## 7 · Management-Summary

In [22]:
total = len(labelled)
labelled_ok = labelled.dropna(subset=["resolution_bucket"])


def share(mask):
    return mask.sum() / total * 100


lines = [
    f"**Datenbasis:** {total:,} Amparex-Tickets (SDAX), "
    f"{labelled['created'].min():%d.%m.%Y} bis {labelled['created'].max():%d.%m.%Y}.",
    "",
    "**Auflösung**",
]
for bucket in BUCKET_ORDER:
    count = (labelled["resolution_bucket"] == bucket).sum()
    lines.append(f"- `{bucket}`: {count:,} Tickets ({count / total * 100:.1f} %)")

lines += ["", "**KI-Eignung**"]
for mode in AI_MODE_ORDER:
    count = (labelled["ai_mode"] == mode).sum()
    lines.append(f"- `{mode}`: {count:,} Tickets ({count / total * 100:.1f} %)")

addressable = (
    labelled["ai_mode"].isin(["fully_automatable", "ai_drafts_human_sends"]).sum()
)
lines += [
    "",
    f"**KI-adressierbar gesamt:** {addressable:,} Tickets "
    f"({addressable / total * 100:.1f} % des Aufkommens) — voll automatisierbar oder "
    "KI-Entwurf mit menschlicher Freigabe.",
    "",
    "**Wo zuerst ansetzen** (Hauptkategorien nach KI-adressierbarem Volumen):",
]
for _, row in category_summary.head(5).iterrows():
    lines.append(
        f"- **{row['Hauptkategorie']}** — {int(row['Tickets']):,} Tickets, "
        f"davon {row['KI_adressierbar']:.0f} % KI-adressierbar "
        f"({int(row['KI_adressierbare_Tickets']):,} Tickets)"
    )

lines += ["", "**Häufigste einzelne Anfragetypen:**"]
for _, row in intents.head(5).iterrows():
    lines.append(
        f"- „{row['intent_label']}\" — {int(row['Tickets']):,} Tickets, "
        f"{row['voll_automatisierbar']:.0f} % voll automatisierbar"
    )

display(Markdown("\n".join(lines)))

**Datenbasis:** 52,695 Amparex-Tickets (SDAX), 01.01.2025 bis 16.09.2026.

**Auflösung**
- `question_answer`: 12,479 Tickets (23.7 %)
- `support_todo`: 16,612 Tickets (31.5 %)
- `team_handoff`: 12,108 Tickets (23.0 %)
- `unclear`: 11,495 Tickets (21.8 %)

**KI-Eignung**
- `fully_automatable`: 9,892 Tickets (18.8 %)
- `ai_drafts_human_sends`: 17,378 Tickets (33.0 %)
- `ai_assisted_triage`: 22,575 Tickets (42.8 %)
- `human_only`: 2,849 Tickets (5.4 %)

**KI-adressierbar gesamt:** 27,270 Tickets (51.8 % des Aufkommens) — voll automatisierbar oder KI-Entwurf mit menschlicher Freigabe.

**Wo zuerst ansetzen** (Hauptkategorien nach KI-adressierbarem Volumen):
- **Abrechnung Kostenträger** — 9,010 Tickets, davon 61 % KI-adressierbar (5,500 Tickets)
- **Kunde** — 6,671 Tickets, davon 69 % KI-adressierbar (4,575 Tickets)
- **Stammdaten** — 6,327 Tickets, davon 67 % KI-adressierbar (4,238 Tickets)
- **Schnittstellen** — 3,930 Tickets, davon 44 % KI-adressierbar (1,737 Tickets)
- **Sonstige Themen** — 4,070 Tickets, davon 33 % KI-adressierbar (1,340 Tickets)

**Häufigste einzelne Anfragetypen:**
- „cloud tse aktivieren" — 355 Tickets, 0 % voll automatisierbar
- „ax startet nicht" — 323 Tickets, 2 % voll automatisierbar
- „rechnung stornieren" — 286 Tickets, 4 % voll automatisierbar
- „dokumentvorlage anpassen" — 284 Tickets, 5 % voll automatisierbar
- „cloud tse einrichten" — 203 Tickets, 3 % voll automatisierbar

In [23]:
# Export fuer Weiterverarbeitung / Weitergabe.
export_columns = [
    "key",
    "created",
    "Hauptkategorie",
    "Unterkategorie",
    "summary",
    "status",
    "resolution_bucket",
    "sub_bucket",
    "intent_label",
    "ai_mode",
    "confidence",
    "reasoning",
    "cloned_to_dev",
    "biz_hours",
]
export = labelled[[c for c in export_columns if c in labelled.columns]]
export.to_csv(
    "data/sdax_ai_assist_classification.csv", sep=";", index=False, encoding="utf-8"
)
category_summary.to_csv(
    "data/sdax_ai_assist_by_category.csv", sep=";", index=False, encoding="utf-8"
)
intents.to_csv(
    "data/sdax_ai_assist_intents.csv", sep=";", index=False, encoding="utf-8"
)
print("Exportiert nach data/sdax_ai_assist_*.csv")

Exportiert nach data/sdax_ai_assist_*.csv
